In [1]:
import sys
sys.path.append("/Users/bayer/embeddings")
import numpy as np
import torch
print(torch.__file__)
import pathlib
import psutil
import gc
import time
import matplotlib.pyplot as plt
from typing import Dict, Any, Optional, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# CEBRA imports
import cebra
from cebra.data import TensorDataset, DatasetCollection
from cebra.data.single_session import SingleSessionDataset
from cebra.data.multi_session import ContinuousMultiSessionDataLoader, MultiSessionLoader
from cebra.models.criterions import InfoNCE, FixedCosineInfoNCE
from cebra.solver.base import Solver

# MNE for EEG data loading
import mne
mne.set_log_level('WARNING')

c:\Users\bayer\miniforge3\envs\cebra\lib\site-packages\torch\__init__.py


In [2]:
import platform
print(platform.architecture())

('64bit', 'WindowsPE')


In [3]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version PyTorch was built with:", torch.version.cuda)
print("Is CUDA available?:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")


PyTorch version: 2.5.1
CUDA version PyTorch was built with: None
Is CUDA available?: False
Device name: N/A


# Dataset

## Directory structure

Each subject (sub-001, sub-002, ...) has their EEG recordings saved in either:
- preproc/ → preprocessed .fif files (e.g., after filtering or cleaning)
- rawdata/ → raw .edf EEG recordings

preproc/
├── sub-001/
│   └── eeg/
│       └── *before_ica.fif
├── sub-002/
│   └── eeg/
│       └── *before_ica.fif
...

rawdata/
├── sub-001/
│   └── eeg/
│       └── *eeg.edf
├── sub-002/
│   └── eeg/
│       └── *eeg.edf
...


# Dataloading

load EEG data for multiple subjects from a structured dataset directory. Each subject's data is in either FIF (preprocessed) or EDF (raw) format.

## Function: `load_subject`

Loads EEG data from a single subject directory.

- Supports **preprocessed FIF files** or **raw EDF files**.
- Returns an `mne.io.Raw` object which encapsulates EEG signals and metadata.
- Raises errors if files are missing or the data type is invalid.
- Used as a building block for loading multi-subject datasets.

**Usage:**  
Called by `load_subjects()` to batch load and preprocess multiple subjects.

## Function: `load_subjects`

Loads multiple subjects from a data directory with options to:

- Pick specific EEG channels (e.g., Fz, Cz).
- Downsample data to reduce memory usage.
- Select a subset of subjects by name.

Returns a dictionary mapping subject IDs to numpy arrays `(samples x channels)`.

**Memory Optimization:**  
Downsampling and picking channels helps keep GPU memory usage manageable during model training.

**Error Handling:**  
Subjects failing to load are skipped with warnings.

## Function: `create_time_index`

Creates continuous time vectors for each subject to enable temporal alignment during contrastive sampling.

- Input: dictionary of EEG numpy arrays `(samples x channels)`.
- Output: dictionary of time arrays `(samples x 1)` in seconds.

This ensures positive pairs sampled during training come from temporally adjacent points.

## Function: `create_combined_cebra_dataset`

Converts per-subject EEG data and time indices into CEBRA `TensorDataset` objects, then combines into a `DatasetCollection`.

- Each session corresponds to one subject’s data.
- `TensorDataset` holds neural data and continuous temporal indices.
- `DatasetCollection` facilitates multi-session contrastive sampling.

This dataset object is essential for CEBRA’s multi-session model training workflow.


In [4]:
def load_subject(subject_folder: pathlib.Path, data_type: str = "preproc") -> mne.io.Raw:
    """
    Load EEG data for a single subject from either preprocessed or raw format.

    Args:
        subject_folder (Path): Path to a subject directory (e.g., sub-001).
        data_type (str): Type of data to load. 
                         - 'preproc': Loads MNE FIF files from the 'eeg' folder.
                         - 'rawdata': Loads EDF files from the 'eeg' folder.

    Returns:
        mne.io.Raw: Loaded EEG data as an MNE Raw object.

    Raises:
        FileNotFoundError: If no valid EEG file is found.
        ValueError: If an invalid data_type is provided.
    """
    eeg_folder = subject_folder / "eeg"
    if data_type == "preproc":
        files = list(eeg_folder.glob("*before_ica.fif"))
        if not files:
            raise FileNotFoundError(f"[INFO] No FIF files found in {eeg_folder}")
        raw = mne.io.read_raw_fif(files[0], preload=True, verbose=False)
    elif data_type == "rawdata":
        files = list(eeg_folder.glob("*.edf"))
        if not files:
            raise FileNotFoundError(f"[INFO] No EDF files found in {eeg_folder}")
        raw = mne.io.read_raw_edf(files[0], preload=True, verbose=False)
    else:
        raise ValueError("data_type must be either 'preproc' or 'rawdata'")
    
    print(f"[INFO] Loaded {files[0].name} for {subject_folder.name}")
    return raw

In [5]:
def load_subjects(
    data_dir: pathlib.Path,
    data_type: str = "preproc",
    pick_channels: Optional[List[str]] = None,
    downsample_factor: int = 1,
    subjects_to_load: Optional[List[str]] = None  # New argument
) -> Dict[str, np.ndarray]:
    """
    Load EEG data with memory monitoring and optimization.
    """
    print(f"=== LOADING SUBJECTS ===")
    
    all_data = {}
    
    for subject_dir in sorted(data_dir.glob("sub-*")):
        subject_key = subject_dir.name
        
        # Skip if subjects_to_load is specified and subject_key not in the list
        if subjects_to_load is not None and subject_key not in subjects_to_load:
            continue
        
        try:
            raw = load_subject(subject_dir, data_type)
            
            # Pick channels if specified
            if pick_channels is not None:
                raw.pick(pick_channels)
            
            # Downsample if needed
            if downsample_factor > 1:
                raw.resample(raw.info['sfreq'] / downsample_factor)
                print(f"[INFO] Downsampled {subject_key} by factor {downsample_factor}")
            
            # Convert to numpy and transpose
            data_np = raw.get_data().T.astype(np.float32)  # (samples x channels)

            all_data[subject_key] = data_np  # Use float32 to save memory
            print(f"[INFO] Loaded {subject_key} with shape {data_np.shape}")
            
            # Clean up
            del raw, data_np
            gc.collect()
            
        except Exception as e:
            print(f"[WARNING] Skipping {subject_key}: {e}")
    
    print(f"[INFO] Total loaded: {len(all_data)} subjects")
    return all_data


In [6]:
def create_time_index(data_dict: Dict[str, np.ndarray], sampling_rate: float = 500.0) -> Dict[str, np.ndarray]:
    """
    Create continuous time index for temporal alignment across sessions.
    """
    time_indices = {}
    
    for subject_key, data in data_dict.items():
        n_samples = data.shape[0]
        # Create time index in seconds
        time_index = np.arange(n_samples) / sampling_rate
        time_indices[subject_key] = time_index.reshape(-1, 1)  # (n_samples, 1)
        
    return time_indices

In [7]:
def create_combined_cebra_dataset(
    data_dict: Dict[str, np.ndarray],
    time_indices: Dict[str, np.ndarray]
) -> cebra.data.DatasetCollection:
    print("=== CREATING COMBINED CEBRA DATASET ===")

    datasets = []

    for session_id, subject_key in enumerate(sorted(data_dict.keys())):
        neural_data = data_dict[subject_key]           # (T, D)
        time_index = time_indices[subject_key]         # (T, 1)

        # Create a TensorDataset for this session with continuous indexing (time)
        session_dataset = cebra.data.datasets.TensorDataset(
            neural=torch.tensor(neural_data, dtype=torch.float32),
            continuous=torch.tensor(time_index, dtype=torch.float32)
        )
        datasets.append(session_dataset)
        print(f"[INFO] Added {subject_key} session dataset: {neural_data.shape}")

    # Combine all session datasets into a DatasetCollection (multi-session dataset)
    dataset_collection = cebra.data.datasets.DatasetCollection(*datasets)
    print(f"[INFO] Created Dataset Collection with {len(datasets)} sessions")
    return dataset_collection

## Load Data

In [8]:
# Load all subjects but keep as Raw objects (no conversion yet)
print("=== LOADING EEG DATA ===")
data_path_preproc = pathlib.Path("preproc_cleaned/preproc")
channels_of_interest = ["Fz", "Cz", "Pz", "Oz"] # reduce data size
subjects = ["sub-001", "sub-002", "sub-003", "sub-004", "sub-005"]
#subjects_data = load_all_subjects(data_dir = data_path_preproc, data_type="preproc", pick_channels=channels_of_interest)
subjects_data = load_subjects(
        data_dir=data_path_preproc,
        data_type="preproc",
        pick_channels=channels_of_interest,
        subjects_to_load=subjects,
        downsample_factor=2  # Downsample by factor of 2 to save memory
    )
time_index = create_time_index(subjects_data)
    
dataset = create_combined_cebra_dataset(subjects_data, time_index)

=== LOADING EEG DATA ===
=== LOADING SUBJECTS ===
[INFO] Loaded sub-001_task-AVR_eeg_preprocessed_filtered_0.1-45_before_ica.fif for sub-001
[INFO] Downsampled sub-001 by factor 2
[INFO] Loaded sub-001 with shape (347968, 4)
[INFO] Loaded sub-002_task-AVR_eeg_preprocessed_filtered_0.1-45_before_ica.fif for sub-002
[INFO] Downsampled sub-002 by factor 2
[INFO] Loaded sub-002 with shape (347964, 4)
[INFO] Loaded sub-003_task-AVR_eeg_preprocessed_filtered_0.1-45_before_ica.fif for sub-003
[INFO] Downsampled sub-003 by factor 2
[INFO] Loaded sub-003 with shape (347964, 4)
[INFO] Loaded sub-004_task-AVR_eeg_preprocessed_filtered_0.1-45_before_ica.fif for sub-004
[INFO] Downsampled sub-004 by factor 2
[INFO] Loaded sub-004 with shape (347964, 4)
[INFO] Loaded sub-005_task-AVR_eeg_preprocessed_filtered_0.1-45_before_ica.fif for sub-005
[INFO] Downsampled sub-005 by factor 2
[INFO] Loaded sub-005 with shape (347964, 4)
[INFO] Total loaded: 5 subjects
=== CREATING COMBINED CEBRA DATASET ===
[IN

# Model 

## Function: `create_model`

Initializes the shared neural network embedding model.

- Input dimension inferred from the dataset.
- Output dimension is the embedding size (e.g., 3 or 8).
- Uses CEBRA's `models.init` to build a feedforward network with specified layers.

The model learns embeddings that maximize similarity of temporally close neural states.

## Function: `create_dataloader`

Generates a `MultiSessionLoader` that samples batches containing:

- Anchors: reference samples.
- Positives: temporally nearby samples from the same session.
- Negatives: samples from different sessions.

This facilitates contrastive learning by providing the required triplets/quartets for InfoNCE loss.

## Function: `train_cebra_model`

Trains the CEBRA model on the combined multi-session dataset.

- Configures model and datasets.
- Uses Adam optimizer with specified learning rate.
- Uses `FixedCosineInfoNCE` criterion for cosine-based contrastive loss.
- Reports training progress and timing.
- Handles exceptions robustly, providing error logs on failure.

Returns training success status, timing, trained models, and solver instance for later evaluation.



In [9]:
# sample batches uniformly across subjects
# sample anchor, positive, and negative pairs from the entire DatasetCollection
def create_dataloader(dataset_collection, batch_size: int, num_steps: int):
    """Create dataloader for multi-session training."""
    print(f"[INFO] Creating multi-session dataloader with batch_size={batch_size}, num_steps={num_steps}")
    
    try:
        dataloader = cebra.data.multi_session.MultiSessionLoader(
            dataset=dataset_collection,
            num_steps=num_steps,
            batch_size=batch_size,
        )
        return dataloader
    except Exception as e:
        print(f"[WARNING] Multi-session dataloader failed: {e}")
        return None


In [10]:
def create_model(dataset_collection: cebra.data.DatasetCollection, output_dim: int = 8):
    print(f"[INFO] Creating model")
    # Get input dimension: uses first subject
    input_dim = dataset_collection.get_input_dimension(session_id=0)    
    print(f"[INFO] Create shared model: {input_dim} -> {output_dim} dims")
    
    # Create single shared model
    model = cebra.models.init(
        name="offset10-model",
        num_neurons=input_dim,
        num_units=256,
        num_output=output_dim
    )

    return model

In [13]:
def train_cebra_model(
    dataset_collection:cebra.data.DatasetCollection,
    device: torch.device,
    num_steps: int = 1000,
    learning_rate: float = 3e-4,
    batch_size: int = 512,
    output_dim: int = 3
) -> Dict[str, Any]:
    """
    Train a shared CEBRA model on the given multi-session DatasetCollection.

    Parameters:
    - dataset_collection: DatasetCollection with all sessions/subjects.
    - device: torch.device to run the model on (cpu or cuda).
    - num_steps: Number of training iterations.
    - learning_rate: Optimizer learning rate.
    - batch_size: Batch size for dataloader.
    - output_dim: Dimension of the learned embedding.

    Returns:
    - dict with training success, training time, trained model, and solver.
    """
                            
    print(f"[INFO] Starting training for {num_steps} steps")
    # Get input dimension from first session
    
    
    input_dim = dataset_collection.get_input_dimension(0)
    model = create_model(dataset_collection, output_dim=output_dim).to(device)

    # Configure dataset for models: sample data so that compatible with the model
    dataset_collection.configure_for(model)
    
    # Create dataloader: returns ContinuousDataLoader which samples:
        # reference samples  
        # positive samples (same subject, nearby in time)  
        # negative samples (cross-subjects)
    dataloader = create_dataloader(dataset_collection, batch_size, num_steps)
    # Initialize solver using CEBRA's solver: forward pass, loss, backprop, optimizer.
    # pulls batches from DataLoader and pushes them to model
    solver = cebra.solver.init(
        name="multi-session",  # shared model for all sessions
        model=model,
        criterion= FixedCosineInfoNCE(temperature=1.0), # temperature scales logit: sharp/smooth distribution
        optimizer=torch.optim.Adam(model.parameters(), lr=learning_rate), # update weights based on gradient
        tqdm_on=True # show progress bar
    )
    
    
    # for measuring training duration
    start_time = time.time()

    
    try:
        print("[INFO] Starting solver.fit()")
        try:
            solver.fit(loader=dataloader)
        except Exception as e_fit:
            print("[ERROR] Exception inside solver.fit():")
            traceback.print_exc()
            raise e_fit  # re-raise to be caught by outer except if needed
        print("[INFO] solver.fit() completed")
        
        # for measuring training duration
        training_time = time.time() - start_time
        
        return {
            'success': True,
            'training_time': training_time,
            'model': model,
            'solver': solver
        }
        
    except Exception as e:
        print(f"[ERROR] Training failed: {e}")
        return {
            'success': False,
            'error': str(e),
            'training_time': time.time() - start_time if 'start_time' in locals() else 0
        }

## Training

In [14]:
from cebra.models.criterions import FixedCosineInfoNCE
import traceback
#  Select device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
try:
    results = train_cebra_model(
        dataset_collection=dataset,
        device=device,
        num_steps=1000,
        learning_rate=3e-4,
        batch_size=256,
        output_dim=3,
    )
except Exception as e:
    print("[FATAL] Uncaught training exception:")
    traceback.print_exc()
    results = None

cpu
[INFO] Starting training for 1000 steps
[INFO] Creating model
[INFO] Create shared model: 4 -> 3 dims
[INFO] Creating multi-session dataloader with batch_size=256, num_steps=1000
[INFO] Starting solver.fit()


pos: -1.0000 neg:  8.1546 total:  7.1546 temperature:  1.0000: 100%|██████████| 1000/1000 [2:21:02<00:00,  8.46s/it]   

[INFO] solver.fit() completed


# Embeddings

In [18]:
if results['success']:
    solver = results['solver']
    dataset_collection = dataset  # Your original dataset collection
    
    # Get embeddings for the dataset_collection
    embeddings = solver.transform(dataset_collection)
    
    print("Embeddings shape:", embeddings.shape)
else:
    print("Training failed, cannot get embeddings.")

RuntimeError: No session_id provided: multisession model requires a session_id to choose the model corresponding to your data shape.

In [ ]:
def generate_embeddings(
    model: torch.nn.Module,  # Single model instead of list
    dataset_collection: DatasetCollection,
    device: torch.device
) -> Dict[str, np.ndarray]:
    """Generate embeddings using the shared model for all sessions."""
    print(f"[INFO] Generating embeddings for {dataset_collection.num_sessions} sessions")
    
    embeddings = {}
    model.eval()
    
    with torch.no_grad():
        for i in range(dataset_collection.num_sessions):
            session_data = dataset_collection.get_session(i)
            
            # Get session data as tensor
            if hasattr(session_data, 'neural'):
                data_tensor = torch.tensor(session_data.neural, dtype=torch.float32).to(device)
            else:
                # Fallback: assume session_data is the tensor itself
                data_tensor = session_data.to(device)
            
            # Generate embeddings
            session_embeddings = model(data_tensor)
            embeddings[f'session_{i}'] = session_embeddings.cpu().numpy()
            
            print(f"[INFO] Generated embeddings for session {i}: {session_embeddings.shape}")
    
    return embeddings

In [ ]:
def create_plotly_visualization(embeddings: Dict[str, np.ndarray]) -> str:
    """Create interactive Plotly visualization of embeddings."""
    print("[INFO] Creating Plotly visualization...")
    
    # Prepare data for plotting
    all_embeddings = []
    all_sessions = []
    all_timepoints = []
    
    for session_key, session_embeddings in embeddings.items():
        if session_embeddings.size > 0:
            n_samples = session_embeddings.shape[0]
            
            all_embeddings.append(session_embeddings)
            all_sessions.extend([session_key] * n_samples)
            all_timepoints.extend(list(range(n_samples)))
    
    if not all_embeddings:
        print("[ERROR] No embeddings to visualize")
        return "<html><body><h1>No embeddings to visualize</h1></body></html>"
    
    # Concatenate all embeddings
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    
    # Create 3D scatter plot using first 3 dimensions
    fig = go.Figure(data=go.Scatter3d(
        x=all_embeddings[:, 0],
        y=all_embeddings[:, 1],
        z=all_embeddings[:, 2] if all_embeddings.shape[1] > 2 else np.zeros(len(all_embeddings)),
        mode='markers',
        marker=dict(
            size=3,
            color=all_timepoints,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Time")
        ),
        text=[f"Session: {s}<br>Time: {t}" for s, t in zip(all_sessions, all_timepoints)],
        hovertemplate='<b>%{text}</b><br>X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title='CEBRA Embeddings - 3D Visualization',
        scene=dict(
            xaxis_title='Embedding Dim 1',
            yaxis_title='Embedding Dim 2',
            zaxis_title='Embedding Dim 3'
        ),
        width=800,
        height=600
    )
    
    return fig.to_html(include_plotlyjs='cdn')

In [ ]:
# Create visualization for the successful training
if results['success']:
    embeddings = results['embedding']
    html_viz = create_plotly_visualization(embeddings)
    